In [1]:
import sys
sys.path.append("..")

import json
import joblib
import pandas as pd

import mlflow
from src.experiment_tracking import setup_tracking, load_production_model
from src.data_loader import load_and_prepare

c:\Users\Twinkle\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(mlflow.__version__)

3.14.0


In [3]:
setup_tracking(tracking_uri="sqlite:///../mlflow.db")
production_model = load_production_model("ckd_prediction_model")
print(type(production_model).__name__)

XGBClassifier


In [4]:
X_train, X_test, y_train, y_test, scaler = load_and_prepare(
    path="../data/processed/kidney_features.csv",
    strategy="smote",
)

feature_order = list(X_train.columns)
print("Number of features:", len(feature_order))
print(feature_order)

Number of features: 26
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'bun_creatinine_ratio', 'anemia_ckd_flag']


In [5]:
from src.models import evaluate_model

scores = evaluate_model(production_model, X_test, y_test)
print(scores)

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0, 'confusion_matrix': [[30, 0], [0, 50]]}


In [6]:
import os
os.makedirs("../models", exist_ok=True)

bundle = {
    "model": production_model,
    "scaler": scaler,
    "feature_order": feature_order,
    "numeric_cols": [
        "age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot",
        "hemo", "pcv", "wc", "rc",
    ],
    "model_name": "xgboost",
    "mlflow_run_id": None,  # filled in Cell 6
}

joblib.dump(bundle, "../models/ckd_pipeline.joblib")
print("Saved bundle to ../models/ckd_pipeline.joblib")

Saved bundle to ../models/ckd_pipeline.joblib


In [7]:
from src.experiment_tracking import get_runs_df, get_best_run

runs_df = get_runs_df()
best_run = get_best_run(runs_df, model_family="xgboost", stage="tuned")

bundle["mlflow_run_id"] = best_run["run_id"]
joblib.dump(bundle, "../models/ckd_pipeline.joblib")
print("Attached run_id:", bundle["mlflow_run_id"])

Attached run_id: b226acef9f824b20943be1718ff39985


In [8]:
fresh_bundle = joblib.load("../models/ckd_pipeline.joblib")

print("Keys:", list(fresh_bundle.keys()))
print("Model type:", type(fresh_bundle["model"]).__name__)
print("Feature count:", len(fresh_bundle["feature_order"]))
print("Run id:", fresh_bundle["mlflow_run_id"])

Keys: ['model', 'scaler', 'feature_order', 'numeric_cols', 'model_name', 'mlflow_run_id']
Model type: XGBClassifier
Feature count: 26
Run id: b226acef9f824b20943be1718ff39985


In [9]:
from src.inference import predict_sample
raw_df = pd.read_csv("../data/processed/kidney_clean.csv")
raw_sample = raw_df.drop(columns=["id", "classification"]).iloc[0].to_dict()
print(raw_sample)

from src.inference import predict_sample
result = predict_sample(raw_sample, bundle_path="../models/ckd_pipeline.joblib")
print(result)


{'age': 48.0, 'bp': 80.0, 'sg': 1.02, 'al': 1.0, 'su': 0.0, 'rbc': 'normal', 'pc': 'normal', 'pcc': 'notpresent', 'ba': 'notpresent', 'bgr': 121.0, 'bu': 36.0, 'sc': 1.2, 'sod': 138.0, 'pot': 4.4, 'hemo': 15.4, 'pcv': 44.0, 'wc': 7800.0, 'rc': 5.2, 'htn': 'yes', 'dm': 'yes', 'cad': 'no', 'appet': 'good', 'pe': 'no', 'ane': 'no'}
{'prediction': 1, 'prediction_label': 'ckd', 'probability': 0.9661610126495361, 'model_name': 'xgboost', 'mlflow_run_id': 'b226acef9f824b20943be1718ff39985'}


In [12]:
from src.features import encode_categoricals, add_domain_features

bundle = joblib.load("../models/ckd_pipeline.joblib")

check_df = pd.DataFrame([raw_sample])
check_df["classification"] = "notckd"  # same placeholder fix
check_df = encode_categoricals(check_df)
check_df = check_df.drop(columns=["classification"])
check_df = add_domain_features(check_df)
check_df[bundle["numeric_cols"]] = bundle["scaler"].transform(check_df[bundle["numeric_cols"]])
check_df = check_df[bundle["feature_order"]]

direct_pred = bundle["model"].predict(check_df)[0]
direct_proba = bundle["model"].predict_proba(check_df)[0][1]

print("predict_sample():", result["prediction"], result["probability"])
print("manual pipeline: ", int(direct_pred), float(direct_proba))

assert result["prediction"] == int(direct_pred)
assert abs(result["probability"] - float(direct_proba)) < 1e-9
print("predict_sample() matches manual preprocessing exactly.")

predict_sample(): 1 0.9661610126495361
manual pipeline:  1 0.9661610126495361
predict_sample() matches manual preprocessing exactly.
